In [0]:

-- 1. Staging view to handle the Window Function imputation and type casting
CREATE LIVE VIEW stg_regions AS
SELECT
  CAST(region_id AS INT) AS region_id,
  sales_district,
  
  -- If sales_region is null, find the MAX(sales_region) for that specific district across all rows
  COALESCE(
    sales_region,
    MAX(sales_region) OVER (PARTITION BY sales_district)
  ) AS sales_region,
  
  ingested_timestamp,
  batch_date
FROM mavenmarket.bronze.regions;

-- 2. Clean Silver Table applying the drop condition
CREATE LIVE TABLE silver_regions (
  -- Rule 1: Drop if region_id is NULL
  CONSTRAINT valid_region_id EXPECT (region_id IS NOT NULL) ON VIOLATION DROP ROW
  
  -- Rule 2: Keep if sales_district is NULL (Achieved by NOT having a drop constraint for it)
)
COMMENT "Cleaned regions dimension with imputed sales regions"
AS SELECT * FROM live.stg_regions;